In [1]:
%pip install -q mlflow scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import mlflow
import pandas as pd
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

print("MLflow:", mlflow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("Imports successful.")

MLflow: 3.17.1
Scikit-learn: 1.6.1
Pandas: 2.2.3
Imports successful.


In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("W11D1_Sklearn_Experiments")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment configured successfully.")

2026/10/13 11:58:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 11:58:58 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 11:59:01 INFO mlflow.tracking.fluent: Experiment with name 'W11D1_Sklearn_Experiments' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Experiment configured successfully.


In [4]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", X.shape[1])

Training rows: 455
Testing rows: 114
Features: 30


In [7]:
def run_experiment(n_estimators, max_depth, min_samples_split):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    with mlflow.start_run():
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        recall = recall_score(y_test, predictions)

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", str(max_depth))
        mlflow.log_param("min_samples_split", min_samples_split)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        mlflow.sklearn.log_model(
    sk_model=model,
    name="model",
    skops_trusted_types=["sklearn.tree._tree.Tree"]
)

        print(
            f"Run: {mlflow.active_run().info.run_id} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f}"
        )

    return model, accuracy

In [8]:
experiment_configs = [
    (50,  None, 2),
    (100, None, 2),
    (150, 10,   2),
    (200, 15,   4),
    (100,  8,   5)
]

experiment_results = []

for config in experiment_configs:
    model, accuracy = run_experiment(*config)

    experiment_results.append({
        "n_estimators": config[0],
        "max_depth": config[1],
        "min_samples_split": config[2],
        "accuracy": accuracy,
        "model": model
    })

print("\nCompleted experiments:", len(experiment_results))

Run: 011f3a97a7714718abbfa3fb9a3e4b4d | Accuracy: 0.9561 | Precision: 0.9589 | Recall: 0.9722
Run: c4cdcf36ab774852875c6b04fde4b29e | Accuracy: 0.9561 | Precision: 0.9589 | Recall: 0.9722
Run: c6cbb34072d64e77acdc2be69c61ca9c | Accuracy: 0.9561 | Precision: 0.9589 | Recall: 0.9722
Run: c6cf52c17aec46df9fcc902bed49fced | Accuracy: 0.9474 | Precision: 0.9583 | Recall: 0.9583
Run: 53fd68c4b69949e4b99eeb791312ff9c | Accuracy: 0.9474 | Precision: 0.9583 | Recall: 0.9583

Completed experiments: 5


In [14]:
# Cell 7: Compare experiments and select the best model

results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "model"}
    for result in experiment_results
])

display(results_df.sort_values("accuracy", ascending=False).reset_index(drop=True))

best_result = max(experiment_results, key=lambda x: x["accuracy"])
best_model = best_result["model"]

print("Best experiment:")
print("Parameters:", {
    "n_estimators": best_result["n_estimators"],
    "max_depth": best_result["max_depth"],
    "min_samples_split": best_result["min_samples_split"]
})
print("Best accuracy:", round(best_result["accuracy"], 4))

,n_estimators,max_depth,min_samples_split,accuracy
0,50,NaN,2,0.956140
1,100,NaN,2,0.956140
2,150,10.0,2,0.956140
3,200,15.0,4,0.947368
4,100,8.0,5,0.947368


Best experiment:
Parameters: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2}
Best accuracy: 0.9561


In [15]:
# Cell 8: Register the best model

from mlflow.models import infer_signature

model_name = "W11D1_Best_RandomForest"

signature = infer_signature(X_train, best_model.predict(X_train))

with mlflow.start_run(run_name="best_model_registration"):
    model_info = mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        signature=signature,
        input_example=X_train.head(5),
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=model_name
)

print("Model registered successfully!")
print("Model name:", registered_model.name)
print("Model version:", registered_model.version)
print("Model URI:", f"models:/{model_name}/{registered_model.version}")

Model registered successfully!
Model name: W11D1_Best_RandomForest
Model version: 1
Model URI: models:/W11D1_Best_RandomForest/1


Successfully registered model 'W11D1_Best_RandomForest'.
Created version '1' of model 'W11D1_Best_RandomForest'.


In [16]:
# Cell 9: Load and test the registered model

model_uri = f"models:/{model_name}/{registered_model.version}"

loaded_model = mlflow.sklearn.load_model(model_uri)

predictions = loaded_model.predict(X_test)

print("Registered model loaded successfully!")
print("Test samples:", len(X_test))
print("First 10 predictions:", predictions[:10])
print("Test accuracy:", round(accuracy_score(y_test, predictions), 4))

Registered model loaded successfully!
Test samples: 114
First 10 predictions: [0 1 0 0 0 1 1 0 0 0]
Test accuracy: 0.9561


In [17]:
# Cell 10: Save experiment results

results_df = results_df.sort_values(
    by="accuracy", ascending=False
).reset_index(drop=True)

results_df.to_csv("w11d1_experiment_results.csv", index=False)

print("Experiment results saved successfully!")
display(results_df)

print("\nCSV file: w11d1_experiment_results.csv")
print("Total experiments:", len(results_df))

assert len(results_df) == 5, "Expected 5 experiments"
print("Validation passed!")

Experiment results saved successfully!


,n_estimators,max_depth,min_samples_split,accuracy
0,50,NaN,2,0.956140
1,100,NaN,2,0.956140
2,150,10.0,2,0.956140
3,200,15.0,4,0.947368
4,100,8.0,5,0.947368



CSV file: w11d1_experiment_results.csv
Total experiments: 5
Validation passed!


In [18]:
# Cell 11: Start MLflow model server

import subprocess
import time
import os

serve_log = open("mlflow_serve.log", "w")

serve_process = subprocess.Popen(
    [
        "mlflow", "models", "serve",
        "-m", model_uri,
        "-p", "5001",
        "--host", "127.0.0.1",
        "--env-manager", "local"
    ],
    stdout=serve_log,
    stderr=subprocess.STDOUT
)

time.sleep(15)

if serve_process.poll() is None:
    print("MLflow model server process is running.")
    print("Endpoint: http://127.0.0.1:5001/invocations")
else:
    serve_log.flush()
    print("Model server stopped. Check the log:")
    print(open("mlflow_serve.log").read()[-4000:])

MLflow model server process is running.
Endpoint: http://127.0.0.1:5001/invocations


In [20]:

# Cell 12: Send a prediction request to the model server

import requests
import time
import pandas as pd

# Use 5 test samples in MLflow's dataframe_split format
payload = {
    "dataframe_split": X_test.head(5).to_dict(orient="split")
}

response = None

for attempt in range(15):
    try:
        response = requests.post(
            "http://127.0.0.1:5001/invocations",
            json=payload,
            timeout=30
        )
        break
    except requests.exceptions.RequestException:
        print(f"Waiting for model server... ({attempt + 1}/15)")
        time.sleep(4)

if response is None:
    print("Could not connect to the model server.")
    print(open("mlflow_serve.log").read()[-4000:])
else:
    print("HTTP status:", response.status_code)
    print(response.text[:2000])
    response.raise_for_status()
    print("Model serving test passed!")

HTTP status: 200
{"predictions": [0, 1, 0, 0, 0]}
Model serving test passed!


In [21]:

# Cell 13: Create submission files

readme = """# W11D1: MLflow Experiment Tracking

## Objective
Track and compare five Random Forest hyperparameter experiments
using MLflow, register the best model, and test model serving.

## Implemented
- MLflow experiment tracking
- Logged model parameters and evaluation metrics
- Saved model artifacts
- Compared five experiments
- Registered and loaded the best model
- Started an MLflow serving endpoint and tested predictions

## Files
- w11d1_experiment_results.csv: Experiment comparison results
- requirements_W11D1.txt: Python dependencies
"""

with open("README_W11D1.md", "w") as f:
    f.write(readme)

requirements = """mlflow
scikit-learn
pandas
requests
"""

with open("requirements_W11D1.txt", "w") as f:
    f.write(requirements)

print("Created README_W11D1.md")
print("Created requirements_W11D1.txt")

Created README_W11D1.md
Created requirements_W11D1.txt


In [22]:

# Cell 14: Validate submission files and experiments

import os

required_files = [
    "w11d1_experiment_results.csv",
    "README_W11D1.md",
    "requirements_W11D1.txt"
]

for filename in required_files:
    print(f"{filename}:",
          "EXISTS" if os.path.exists(filename) else "MISSING")

print("\nExperiments recorded:", len(experiment_results))
print("Registered model:", model_name)
print("Registered version:", registered_model.version)
print("Loaded model test accuracy:",
      round(accuracy_score(y_test, predictions), 4))

assert len(experiment_results) == 5
assert all(os.path.exists(f) for f in required_files)

print("\nW11D1 validation completed!")


w11d1_experiment_results.csv: EXISTS
README_W11D1.md: EXISTS
requirements_W11D1.txt: EXISTS

Experiments recorded: 5
Registered model: W11D1_Best_RandomForest
Registered version: 1
Loaded model test accuracy: 0.9561

W11D1 validation completed!
